In [45]:
from dotenv import load_dotenv
load_dotenv()

True

In [61]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate   ## in notebook 2 we hv used ChatPromptTemplate, since that time we are creating Template of chat messages
from langchain_core.output_parsers import StrOutputParser

In [47]:
loader = PyPDFLoader("../data/data_science_syllabus.pdf")  
docs = loader.load()
len(docs)

10

In [48]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap=200)
splitted_data = splitter.split_documents(docs)

In [49]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")
vector_store = Chroma.from_documents(    #yaha hmne data ko disk mei persist nhi kiya , data abhi issi variable mei hi hai
    documents= splitted_data,
    embedding= embeddings,
)

In [50]:
query = "Machine Learning and Data Science Content"
data = vector_store.similarity_search(query)
# len(data)
print(data[0])

page_content='📘  1Year Roadmap: Data Analytics, Data Science & GenAI
10' metadata={'moddate': '2025-12-13T08:07:26+00:00', 'source': '../data/data_science_syllabus.pdf', 'page_label': '10', 'page': 9, 'producer': 'Skia/PDF m137', 'creationdate': '2025-12-13T08:07:26+00:00', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/137.0.0.0 Safari/537.36', 'title': '📘 1-Year Roadmap: Data Analytics, Data Science & GenAI', 'total_pages': 10}


In [51]:
context = ""
for doc in data:                    ## building context for the llm
    context += doc.page_content + '\n'

In [52]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
res = llm.invoke(f""" Can you provide me ans based on provided
context for my question, context: {context} and question: {query} """)

In [53]:
print(res.content[0]["text"])

Based on the provided context, the text only mentions the title: **"📘 1-Year Roadmap: Data Analytics, Data Science & GenAI"** and the number **"10"**. 

It does not contain any specific details regarding "Machine Learning and Data Science Content." Therefore, based on the context provided, no further information is available to answer your question.


### Build Chain - context_generater | prompt | llm | strParser

In [54]:
def get_context(query:str):
    data = vector_store.similarity_search(query)
    context = ""
    for doc in data:                    
        context += doc.page_content + '\n'

    return {
        "context" : context,
        "question": query
    }

In [55]:
prompt = PromptTemplate.from_template(
    """
    You are a helpful assistant and provide answer based on the context of user question
    if u don't know the answer, then u can say "i don't know"
    Context: {context}
    Question: {question}
"""
)

In [ ]:
out = StrOutputParser()   ## to get direct and clear output
rag_chain = get_context | prompt | llm |out

In [63]:
res = rag_chain.invoke("What is duration of module 3??")
print(res)

Based on the provided context, the duration of Module 3 is 3 weeks (Month 3).
